In [1]:
# install
!pip install pmdarima kagglehub tqdm requests --quiet

In [2]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import warnings
import os
from datetime import datetime
from tqdm.auto import tqdm
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')

COINS = [
    'BTC-USD', 'ETH-USD', 'XRP-USD', 'BNB-USD', 'SOL-USD',
    'TRX-USD', 'DOGE-USD', 'ADA-USD', 'HYPE-USD', 'BCH-USD',
    'LINK-USD', 'LEO-USD', 'ZEC-USD', 'XLM-USD', 'XMR-USD',
    'LTC-USD', 'HBAR-USD', 'AVAX-USD'
]

TRAIN_DAYS = 730
TEST_DAYS = 365
HORIZONS = [1, 7]
REFIT_EVERY = 60

OUTPUT_DIR = '/content/arimax_output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"ARIMAX for {len(COINS)} coins")


ARIMAX for 20 coins


In [3]:
# fear and greed index

def get_fear_greed():
    """get historical fear and greed from alternative.me"""
    url = "https://api.alternative.me/fng/"
    params = {'limit': 0, 'format': 'json'}  # 0 = all data

    try:
        resp = requests.get(url, params=params, timeout=30)
        data = resp.json()['data']

        fng = pd.DataFrame(data)
        fng['date'] = pd.to_datetime(fng['timestamp'].astype(int), unit='s')
        fng['fng'] = fng['value'].astype(int)
        fng = fng[['date', 'fng']].sort_values('date').reset_index(drop=True)

        print(f"Fear & Greed: {len(fng)} days")
        return fng
    except Exception as e:
        print(f"FNG fetch failed: {e}")
        return None

fng_data = get_fear_greed()

Fear & Greed: 2859 days


In [4]:
# download price data
import kagglehub
import glob

path = kagglehub.dataset_download("isaaclopgu/cryptocurrency-historical-prices-top-100-2025")
files = glob.glob(f"{path}/*.csv") + glob.glob(f"{path}/**/*.csv", recursive=True)
DATA_FILE = files[0]
print(f"Price data: {DATA_FILE}")

Using Colab cache for faster access to the 'cryptocurrency-historical-prices-top-100-2025' dataset.
Price data: /kaggle/input/cryptocurrency-historical-prices-top-100-2025/Crypto_historical_data.csv


In [5]:
# feature engineering

def compute_rsi(series, period=14):
    """relative strength index"""
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def compute_atr(high, low, close, period=14):
    """average true range"""
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def build_features(filepath, fng_df):

    # load data
    df = pd.read_csv(filepath)
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None)
    df['Date'] = pd.to_datetime(df['Date'].dt.date)

    df = df[df['ticker'].isin(COINS)].copy()

    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['Open', 'High', 'Low', 'Close'])

    # fix ohlc
    df['High'] = df[['Open', 'High', 'Low', 'Close']].max(axis=1)
    df['Low'] = df[['Open', 'High', 'Low', 'Close']].min(axis=1)

    df = df.sort_values(['ticker', 'Date', 'Volume'], ascending=[True, True, False])
    df = df.drop_duplicates(subset=['ticker', 'Date'], keep='first')
    df = df.sort_values(['ticker', 'Date']).reset_index(drop=True)

    # target
    df['PriceRange'] = df['High'] - df['Low']
    df['LogRange'] = np.log(df['PriceRange'].replace(0, np.nan))

    # log returns
    df['log_return'] = df.groupby('ticker')['Close'].transform(lambda x: np.log(x / x.shift(1)))

    # rolling volatility
    for w in [5, 10, 20, 30]:
        df[f'volatility_{w}d'] = df.groupby('ticker')['log_return'].transform(
            lambda x: x.rolling(w, min_periods=1).std()
        )

    # simple moving averages
    for w in [5, 20, 50, 200]:
        df[f'sma_{w}'] = df.groupby('ticker')['Close'].transform(
            lambda x: x.rolling(w, min_periods=1).mean()
        )

    # price deviation from SMAs
    df['price_dev_sma20'] = (df['Close'] - df['sma_20']) / df['sma_20']
    df['price_dev_sma50'] = (df['Close'] - df['sma_50']) / df['sma_50']

    # RSI
    df['rsi_14'] = df.groupby('ticker')['Close'].transform(lambda x: compute_rsi(x, 14))

    # MACD
    df['ema_12'] = df.groupby('ticker')['Close'].transform(lambda x: x.ewm(span=12).mean())
    df['ema_26'] = df.groupby('ticker')['Close'].transform(lambda x: x.ewm(span=26).mean())
    df['macd'] = df['ema_12'] - df['ema_26']
    df['macd_signal'] = df.groupby('ticker')['macd'].transform(lambda x: x.ewm(span=9).mean())
    df['macd_hist'] = df['macd'] - df['macd_signal']

    # rate of change
    df['roc_10'] = df.groupby('ticker')['Close'].transform(lambda x: x.pct_change(10) * 100)
    df['roc_20'] = df.groupby('ticker')['Close'].transform(lambda x: x.pct_change(20) * 100)

    # bollinger bands
    df['bb_middle'] = df['sma_20']
    df['bb_std'] = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(20, min_periods=1).std())
    df['bb_upper'] = df['bb_middle'] + 2 * df['bb_std']
    df['bb_lower'] = df['bb_middle'] - 2 * df['bb_std']
    df['bb_position'] = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

    # volume ratios
    for w in [5, 10, 20]:
        vol_ma = df.groupby('ticker')['Volume'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'vol_ratio_{w}d'] = df['Volume'] / vol_ma

    df['vol_pct_change'] = df.groupby('ticker')['Volume'].pct_change()

    # ATR
    df['atr_14'] = df.groupby('ticker').apply(
        lambda g: compute_atr(g['High'], g['Low'], g['Close'], 14)
    ).reset_index(level=0, drop=True)
    df['atr_21'] = df.groupby('ticker').apply(
        lambda g: compute_atr(g['High'], g['Low'], g['Close'], 21)
    ).reset_index(level=0, drop=True)

    # lagged returns
    for lag in [1, 2, 3, 5, 7]:
        df[f'return_lag{lag}'] = df.groupby('ticker')['log_return'].shift(lag)

    # lagged volume
    for lag in [1, 2, 3, 5]:
        df[f'vol_lag{lag}'] = df.groupby('ticker')['Volume'].shift(lag)

    # fear and greed - external sentiment signal
    if fng_df is not None:
        fng = fng_df.rename(columns={'date': 'Date'})
        df = df.merge(fng[['Date', 'fng']], on='Date', how='left')
        df['fng'] = df['fng'].fillna(method='ffill').fillna(50)  # neutral if missing
        df['fng_ma7'] = df.groupby('ticker')['fng'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    else:
        df['fng'] = 50
        df['fng_ma7'] = 50

    # BTC market leader
    btc = df[df['ticker'] == 'BTC-USD'][['Date', 'log_return', 'volatility_5d']].copy()
    btc.columns = ['Date', 'btc_return', 'btc_vol']
    df = df.merge(btc, on='Date', how='left')

    # clean up temp columns and infinities
    df = df.drop(columns=['ema_12', 'ema_26', 'bb_middle', 'bb_std'], errors='ignore')
    df = df.replace([np.inf, -np.inf], np.nan)

    print(f"Built {len([c for c in df.columns if c not in ['Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'PriceRange', 'LogRange']])} features")
    print(f"Data shape: {df.shape}")

    return df

df = build_features(DATA_FILE, fng_data)
df.to_csv(f"{OUTPUT_DIR}arimax_features.csv", index=False)

Built 40 features
Data shape: (53042, 49)


In [6]:
# feature selection

FEATURES = [
    # momentum/returns
    'log_return',
    'return_lag1',
    'rsi_14',
    'macd_hist',

    # volatility
    'volatility_20d',
    'atr_14',

    # trend
    'price_dev_sma20',
    'bb_position',

    # volume
    'vol_ratio_5d',

    # external
    'fng',
    'btc_return',
    'btc_vol',
]

print(f"Using {len(FEATURES)} exogenous features")

Using 12 exogenous features


In [7]:
# ARIMAX class

class CryptoARIMAX:
    def __init__(self, train_days, test_days, horizons, refit_every, features):
        self.train_days = train_days
        self.test_days = test_days
        self.horizons = horizons
        self.refit_every = refit_every
        self.features = features

    def prepare_data(self, df, ticker):
        """get data for one coin, lag features by 1 day"""
        coin = df[df['ticker'] == ticker].copy()
        coin = coin.sort_values('Date').reset_index(drop=True)

        # lag features by 1 to prevent lookahead
        for f in self.features:
            if f in coin.columns:
                coin[f'{f}_lag1'] = coin[f].shift(1)

        # drop first row
        coin = coin.iloc[1:].reset_index(drop=True)
        return coin

    def get_feature_cols(self):
        return [f'{f}_lag1' for f in self.features]

    def find_order(self, y, X):
        try:
            model = auto_arima(
                y, exogenous=X,
                start_p=0, max_p=3, start_q=0, max_q=3,
                d=None, max_d=2, seasonal=False, stepwise=True,
                suppress_warnings=True, error_action='ignore',
                information_criterion='bic'
            )
            return model.order
        except:
            return (1, 1, 1)

    def run(self, df, ticker):
        coin = self.prepare_data(df, ticker)
        feat_cols = self.get_feature_cols()

        n = len(coin)
        test_size = min(self.test_days, n - self.train_days)

        if test_size < 30:
            print(f"  {ticker}: not enough data")
            return None

        test_start = n - test_size
        print(f"  {ticker}: n={n}, testing idx {test_start} to {n}")

        init_data = coin.iloc[:test_start]
        y_init = init_data['LogRange'].dropna()
        X_init = init_data.loc[y_init.index, feat_cols].fillna(0)

        order = self.find_order(y_init.tail(365), X_init.tail(365))
        print(f"    order: {order}")

        results = {h: [] for h in self.horizons}
        refit_counter = 0
        expected = {h: 0 for h in self.horizons}

        for i in tqdm(range(test_start, n), desc=f"    {ticker}", leave=False):
            refit_counter += 1

            # rolling training window
            start = max(0, i - self.train_days)
            train = coin.iloc[start:i]
            y_train = train['LogRange'].dropna()

            if len(y_train) < 50:
                continue

            X_train = train.loc[y_train.index, feat_cols].fillna(0)

            # refit
            if refit_counter >= self.refit_every:
                order = self.find_order(y_train.tail(365), X_train.tail(365))
                refit_counter = 0

            # fit ARIMAX
            try:
                model = ARIMA(y_train, exog=X_train, order=order)
                fit = model.fit()
            except:
                continue

            # forecast
            for h in self.horizons:
                target_idx = i + h - 1
                expected[h] += 1

                if target_idx >= n:
                    continue

                actual = coin.iloc[target_idx]['LogRange']
                if pd.isna(actual):
                    continue

                try:
                    X_future = coin.iloc[i:i+h][feat_cols].fillna(0).values

                    fc = fit.get_forecast(steps=h, exog=X_future)
                    pred = fc.predicted_mean.iloc[-1]

                    # statsmodels intervals
                    ci80 = fc.conf_int(alpha=0.20).iloc[-1]
                    ci95 = fc.conf_int(alpha=0.05).iloc[-1]

                    results[h].append({
                        'date': coin.iloc[target_idx]['Date'],
                        'actual': actual,
                        'predicted': pred,
                        'lower_80': ci80.iloc[0],
                        'upper_80': ci80.iloc[1],
                        'lower_95': ci95.iloc[0],
                        'upper_95': ci95.iloc[1]
                    })
                except:
                    continue

        for h in self.horizons:
            results[h] = pd.DataFrame(results[h]) if results[h] else pd.DataFrame()

        completion = {h: len(results[h]) / max(expected[h], 1) for h in self.horizons}

        return {'ticker': ticker, 'order': order, 'results': results, 'completion': completion}

    def calc_metrics(self, res_df):
        if res_df.empty:
            return {}

        actual = res_df['actual'].values
        pred = res_df['predicted'].values

        mask = np.isfinite(actual) & np.isfinite(pred)
        actual, pred = actual[mask], pred[mask]

        if len(actual) == 0:
            return {}

        errors = actual - pred

        m = {
            'n': len(actual),
            'rmse': np.sqrt(np.mean(errors**2)),
            'mae': np.mean(np.abs(errors)),
            'bias': np.mean(errors)
        }

        # mape
        with np.errstate(divide='ignore', invalid='ignore'):
            ape = np.abs(errors / actual) * 100
            ape = ape[np.isfinite(ape)]
            m['mape'] = np.mean(ape) if len(ape) > 0 else np.nan

        # coverage
        l80, u80 = res_df['lower_80'].values[mask], res_df['upper_80'].values[mask]
        l95, u95 = res_df['lower_95'].values[mask], res_df['upper_95'].values[mask]

        m['coverage_80'] = np.mean((actual >= l80) & (actual <= u80)) * 100
        m['coverage_95'] = np.mean((actual >= l95) & (actual <= u95)) * 100
        m['width_80'] = np.mean(u80 - l80)
        m['width_95'] = np.mean(u95 - l95)

        return m

print("ARIMAX class ready")

ARIMAX class ready


In [ ]:
# run ARIMAX
model = CryptoARIMAX(TRAIN_DAYS, TEST_DAYS, HORIZONS, REFIT_EVERY, FEATURES)
all_results = {}
all_metrics = []

for ticker in COINS:
    print(f"\n[{COINS.index(ticker)+1}/{len(COINS)}] {ticker}")

    try:
        result = model.run(df, ticker)
        if result is None:
            continue

        all_results[ticker] = result

        for h in HORIZONS:
            res_df = result['results'][h]
            if not res_df.empty:
                m = model.calc_metrics(res_df)
                m['ticker'] = ticker
                m['horizon'] = h
                m['order'] = str(result['order'])
                m['completion'] = result['completion'][h]
                all_metrics.append(m)

    except Exception as e:
        print(f"  error: {e}")

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(f"{OUTPUT_DIR}arimax_metrics.csv", index=False)

print("ARIMAX Results")
print(metrics_df.round(3).to_string(index=False))


[1/20] BTC-USD
  BTC-USD: n=4094, testing idx 3729 to 4094
    order: (0, 1, 2)


    BTC-USD:   0%|          | 0/365 [00:00<?, ?it/s]


[2/20] ETH-USD
  ETH-USD: n=2945, testing idx 2580 to 2945
    order: (1, 0, 2)


    ETH-USD:   0%|          | 0/365 [00:00<?, ?it/s]


[3/20] XRP-USD
  XRP-USD: n=2945, testing idx 2580 to 2945
    order: (1, 0, 2)


    XRP-USD:   0%|          | 0/365 [00:00<?, ?it/s]


[4/20] BNB-USD
  BNB-USD: n=2945, testing idx 2580 to 2945
    order: (1, 1, 1)


    BNB-USD:   0%|          | 0/365 [00:00<?, ?it/s]

In [ ]:
# plots

def plot_coin(result, ticker, output_dir):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for idx, h in enumerate(HORIZONS):
        res = result['results'][h]
        if res.empty:
            continue

        ax = axes[idx, 0]

        ax.fill_between(res['date'], res['lower_95'], res['upper_95'],
                       alpha=0.15, color='blue', label='95% CI')
        ax.fill_between(res['date'], res['lower_80'], res['upper_80'],
                       alpha=0.25, color='blue', label='80% CI')
        ax.plot(res['date'], res['actual'], 'k.-', ms=2, alpha=0.6, label='Actual')
        ax.plot(res['date'], res['predicted'], 'r-', lw=1, alpha=0.8, label='ARIMAX')

        ax.set_title(f'{ticker} - {h}-Day ARIMAX')
        ax.set_ylabel('Log(Price Range)')
        ax.legend(loc='upper right', fontsize=8)
        ax.tick_params(axis='x', rotation=30)

        ymin, ymax = ax.get_ylim()
        padding = (ymax - ymin) * 0.1
        ax.set_ylim(ymin - padding, ymax + padding)

        # errors
        ax2 = axes[idx, 1]
        errors = res['actual'] - res['predicted']
        ax2.hist(errors, bins=25, alpha=0.7, edgecolor='black', density=True)
        ax2.axvline(0, color='red', linestyle='--')
        ax2.axvline(errors.mean(), color='green', label=f'mean={errors.mean():.3f}')
        ax2.set_title(f'{h}-Day Errors')
        ax2.set_xlabel('Error')
        ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"{output_dir}{ticker.replace('-', '_')}_arimax.png", dpi=120, bbox_inches='tight')
    plt.close()

for ticker, result in all_results.items():
    plot_coin(result, ticker, OUTPUT_DIR)

print(f"Plots saved to {OUTPUT_DIR}")

# summary plot
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# mape
ax1 = axes[0, 0]
h1 = metrics_df[metrics_df['horizon'] == 1].sort_values('mape')
ax1.barh(h1['ticker'], h1['mape'], color='steelblue', alpha=0.8)
ax1.set_xlabel('MAPE (%)')
ax1.set_title('1-Day ARIMAX MAPE')
ax1.axvline(h1['mape'].median(), color='red', linestyle='--', alpha=0.7)

# coverage
ax2 = axes[0, 1]
h1 = metrics_df[metrics_df['horizon'] == 1]
x = range(len(h1))
ax2.scatter(x, h1['coverage_80'], label='80% cov', s=50)
ax2.scatter(x, h1['coverage_95'], label='95% cov', s=50, marker='s')
ax2.axhline(80, color='orange', linestyle='--', alpha=0.6)
ax2.axhline(95, color='red', linestyle='--', alpha=0.6)
ax2.set_xticks(x)
ax2.set_xticklabels(h1['ticker'], rotation=45, ha='right', fontsize=7)
ax2.set_ylabel('Coverage (%)')
ax2.set_title('1-Day Interval Coverage')
ax2.legend(fontsize=8)

# rmse
ax3 = axes[1, 0]
metrics_df.boxplot(column='rmse', by='horizon', ax=ax3)
ax3.set_title('RMSE by Horizon')
ax3.set_xlabel('Horizon')
ax3.set_ylabel('RMSE')
plt.suptitle('')

# bias
ax4 = axes[1, 1]
h1 = metrics_df[metrics_df['horizon'] == 1]
colors = ['green' if b > 0 else 'red' for b in h1['bias']]
ax4.bar(range(len(h1)), h1['bias'], color=colors, alpha=0.7)
ax4.set_xticks(range(len(h1)))
ax4.set_xticklabels(h1['ticker'], rotation=45, ha='right', fontsize=7)
ax4.axhline(0, color='black', lw=1)
ax4.set_ylabel('Bias')
ax4.set_title('1-Day Bias')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}arimax_summary.png", dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# summary
print("ARIMAX Sumary")

for h in HORIZONS:
    hdata = metrics_df[metrics_df['horizon'] == h]
    print(f"\n{h}-Day Horizon:")
    print(f"  Coins: {len(hdata)}")
    print(f"  Total predictions: {hdata['n'].sum()}")
    print(f"  Mean MAPE: {hdata['mape'].mean():.1f}%")
    print(f"  Mean RMSE: {hdata['rmse'].mean():.3f}")
    print(f"  80% Coverage: {hdata['coverage_80'].mean():.1f}%")
    print(f"  95% Coverage: {hdata['coverage_95'].mean():.1f}%")
    print(f"  Completion: {hdata['completion'].mean():.1%}")

print(f"\nFeatures used: {len(FEATURES)}")
for f in FEATURES:
    print(f"  - {f}")


In [ ]:
# download

from google.colab import files
import shutil

shutil.make_archive('/content/arimax_results', 'zip', OUTPUT_DIR)
files.download('/content/arimax_results.zip')